# Run the NLI comparisons

Use the same dataset hash, track, partitions and windows for TF-IDF logistic regression, BiLSTM, a frozen general-domain checkpoint and fine-tuned legal DeBERTa. Runs export probability scores and per-class metrics.

Sources: [ContractNLI](https://aclanthology.org/2021.findings-emnlp.164/), [DeBERTa-v3](https://arxiv.org/abs/2111.09543) and [dataset preparation](https://stanfordnlp.github.io/contract-nli/). Persist outputs from the Colab VM before ending the session.

In [ ]:
import os, subprocess, sys, json
from pathlib import Path
# Upload or mount the upgrade branch, then point this at its path in Colab.
PROJECT = Path('/content/LegalShield')
assert (PROJECT / 'research' / 'requirements.txt').exists(), 'Set PROJECT to the checked-out upgrade branch'
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'research/requirements.txt'], check=True)
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU runtime')
def run(module, *args):
    subprocess.run([sys.executable, '-m', module, *map(str, args)], check=True)

In [ ]:
run('research.nli', 'tfidf', 'artifacts/data/contractnli.jsonl', 'artifacts/tfidf')
run('research.nli', 'general', 'artifacts/data/contractnli.jsonl', 'artifacts/general')
for name in ('tfidf', 'bilstm', 'general', 'deberta'):
    folder = Path('artifacts') / name
    print(name, json.loads((folder / 'metrics.json').read_text())['macro'])
    run('research.plots', 'nli', folder, folder / 'figures')